In [1]:
import pandas as pd

# Test student/worker agents
Some agents show up as a student in one school and a worker in another. This is possible because agents over 16 can be workers. IPF may assign them commuter_workplace_category == 10, which corresponds to NAICS category 61 (Education services). Then they may be assigned to one school as a student because they are 16 and then assigned as a worker in another school because commuter_workplace_category == 10. When you visualize the school network with something like Gephi, this appears as a between-school connections when that agent (node) is present in school A and also school B. 

**Other notes:**
* Agents who work in schools are listed in `pop_export/sch_workers.csv` and are present in the school network `pop_export/starsim/net_s.csv`. They are not present in the workplace network `net_w.csv`
* School grades come directly from PUMS data. So, if there are older ages in younger grades (eg, a 50-year old who is in 8th grade), that is technically possible if it’s in the PUMS data.
* If a student-worker does not appear as a school worker or a student in the school network, and they do not appear in the work network, then they are a worker in the group quarters network `net_g.csv`. There should not be any student-worker agents who are not in either the school, work, or GQ networks.

The following code tests this...

In [2]:
# Read in all agents and attributes (for Howard County, MD)
# print grades represented in people_all
people_all = pd.read_csv('pops/sc_45083/pop_export/people_all.csv')
people_all.sch_grade.unique()

/var/folders/9n/1361ppx53h5cjy3q80rwhj540000gn/T/ipykernel_6333/2258394403.py:3: DtypeWarning: Columns (0: sch_grade) have mixed types. Specify dtype option on import or set low_memory=False.
  people_all = pd.read_csv('pops/sc_45083/pop_export/people_all.csv')


<ArrowStringArray>
[ nan,  'g',  'c',  '6',  '3',  '1',  'k', '10',  '8',  '4',  '7',  'p',  '2',
 '11',  '5',  '9', '12']
Length: 17, dtype: str

In [3]:
# Filter for students who are also working
student_workers = people_all.loc[(~people_all['sch_grade'].isnull()) & (people_all['working'] == 1)]
student_workers = student_workers[['uid', 'sch_grade', 'age', 'commuter_workplace_category']]

# Group by grade and age, print grades of student workers
student_workers_counts = student_workers.groupby(['sch_grade', 'age']).size().reset_index(name='count')
student_workers_counts.sch_grade.unique() # student workers are only in older grades


<ArrowStringArray>
['10', '11', '12', '8', '9', 'c', 'g']
Length: 7, dtype: str

In [4]:
# Print age counts by grade
student_workers_age = student_workers.groupby(['sch_grade', 'age']).size().reset_index(name='count')
student_workers_age
# There are older ages among student workers in these grades

,sch_grade,age,count
0,10,16.0,284
1,10,17.0,28
2,10,25.0,25
3,10,60.0,16
4,11,16.0,504
...,...,...,...
94,g,54.0,24
95,g,56.0,51
96,g,65.0,10
97,g,66.0,16


In [ ]:
# Now see how many of these students are also workers in education (have commuter_workplace_category == 10, which corresponds to NAICS code 61 "Educational services and health care and social assistance:Educational services")
student_workers_sch = student_workers.loc[student_workers['commuter_workplace_category'] == 10]
student_workers_sch
# There are 1098 students with commuter_workplace_category == 10, which means they may be assigned as school workers in workplaces.py

,uid,sch_grade,age,commuter_workplace_category
1,1,g,50.0,10.0
12,12,c,23.0,10.0
239,239,9,20.0,10.0
336,336,g,50.0,10.0
867,867,g,50.0,10.0
...,...,...,...,...
294622,294622,g,22.0,10.0
295235,295235,12,53.0,10.0
296099,296099,g,37.0,10.0
296505,296505,g,37.0,10.0


In [ ]:
# Now see how many of these agents show up in more than one school, once as a worker and once as a student

# First, get uids of students by school
sch_students = pd.read_csv('pops/sc_45083/pop_export/sch_students.csv')
sch_students = sch_students.merge(
    people_all[['uid', 'p_id', 'hh_id', 'cbg_id']],
    on=['p_id', 'hh_id', 'cbg_id'],
    how='left',
)
sch_students

,sch_code,p_id,hh_id,cbg_id,uid
0,450348000060,3,2,51,68590
1,450348000060,3,40,51,68676
2,450348000060,4,42,51,68683
3,450348000060,2,44,51,68690
4,450348000060,4,47,51,68697
...,...,...,...,...,...
55110,450390101722,4,732,175,274244
55111,450390101722,3,764,175,274340
55112,450390101722,2,3,189,291385
55113,450390101722,3,220,193,295220


In [ ]:
# Now, get uids of workers by school
sch_workers = pd.read_csv('pops/sc_45083/pop_export/sch_workers.csv')
sch_workers = sch_workers.merge(
    people_all[['uid', 'p_id', 'hh_id', 'cbg_id']],
    on=['p_id', 'hh_id', 'cbg_id'],
    how='left',
)
sch_workers

,sch_code,p_id,hh_id,cbg_id,uid
0,450348000060,37610,0,0,334272
1,450348000060,1,290,63,87527
2,450348000060,2,128,86,126867
3,450348000060,37611,0,0,334273
4,450348000060,37612,0,0,334274
...,...,...,...,...,...
3453,450390101722,2,422,180,279028
3454,450390101722,2,913,173,269863
3455,450390101722,1,477,38,45073
3456,450390101722,39092,0,0,335754


In [ ]:
# See agents who are students and workers in the same school
same_school = (
    sch_students[['uid', 'sch_code']]
    .dropna(subset=['uid'])
    .merge(
        sch_workers[['uid', 'sch_code']].dropna(subset=['uid']),
        on=['uid', 'sch_code'],
        how='inner',
    )
    .drop_duplicates()
)
print(f"uids in the same school as student and worker: {same_school['uid'].nunique()}")
same_school # there are five agents who are a student and a worker in the same school

uids in the same school as student and worker: 5


,uid,sch_code
0,31130,450360001022
1,55324,450360001022
2,60136,450363001038
3,213134,450363001038
4,191127,450366001052


In [ ]:
# Look at attributes of those agents
same_school_agents = people_all.loc[people_all['uid'].isin(same_school['uid'])]
same_school_agents


,uid,p_id,hh_id,cbg_id,sample_index,state,county,tract,cbg_geocode,household,...,race_pacific_alone,race_other_alone,race_two_or_more,hispanic,working,commuter,commuter_income_category,commuter_workplace_category,sch_grade,sch_code
31130,31130,4,343,26,230407.0,45.0,45083.0,4.508302e+10,4.508302e+11,11907.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1.0,10.0,12,4.503600e+11
55324,55324,3,813,42,185551.0,45.0,45083.0,4.508302e+10,4.508302e+11,21493.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,10.0,11,4.503600e+11
60136,60136,3,430,45,84256.0,45.0,45083.0,4.508302e+10,4.508302e+11,23364.0,...,0.0,0.0,0.0,0.0,1.0,1.0,2.0,10.0,9,4.503630e+11
191127,191127,3,278,124,84256.0,45.0,45083.0,4.508302e+10,4.508302e+11,74456.0,...,0.0,0.0,0.0,0.0,1.0,1.0,2.0,10.0,9,4.503660e+11
213134,213134,1,570,140,21493.0,45.0,45083.0,4.508302e+10,4.508302e+11,82866.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,10.0,12,4.503630e+11


In [10]:
# Now see which agents are in one school as a student and another as a worker
diff_school = (
    sch_students[['uid', 'sch_code']]
    .dropna(subset=['uid'])
    .rename(columns={'sch_code': 'sch_code_student'})
    .merge(
        sch_workers[['uid', 'sch_code']]
        .dropna(subset=['uid'])
        .rename(columns={'sch_code': 'sch_code_worker'}),
        on='uid',
        how='inner',
    )
)
diff_school = diff_school[
    diff_school['sch_code_student'] != diff_school['sch_code_worker']
].drop_duplicates()

print(f"uids in one school as student and another as worker: {diff_school['uid'].nunique()}")
diff_school

uids in one school as student and another as worker: 9


,uid,sch_code_student,sch_code_worker
0,35922,450348000988,450351001558
1,259863,450348000988,450390101639
2,70547,450348000989,450366001042
3,71901,450351000937,450354001011
4,67093,450354001010,450354001005
9,239,450366001052,450366001046
10,42152,450366001052,450366001051
12,246537,450366001052,450360001022
13,26352,450390101639,450360001022


In [11]:
# Look at attributes of those agent
diff_school_agents = people_all.loc[people_all['uid'].isin(diff_school['uid'])]
diff_school_agents

,uid,p_id,hh_id,cbg_id,sample_index,state,county,tract,cbg_geocode,household,...,race_pacific_alone,race_other_alone,race_two_or_more,hispanic,working,commuter,commuter_income_category,commuter_workplace_category,sch_grade,sch_code
239,239,3,88,1,84256.0,45.0,45083.0,4.508302e+10,4.508302e+11,88.0,...,0.0,0.0,0.0,0.0,1.0,1.0,2.0,10.0,9,4.503660e+11
26352,26352,4,229,23,230407.0,45.0,45083.0,4.508302e+10,4.508302e+11,10016.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1.0,10.0,12,4.503901e+11
35922,35922,3,294,30,185551.0,45.0,45083.0,4.508302e+10,4.508302e+11,13863.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,10.0,11,4.503480e+11
42152,42152,1,672,35,21493.0,45.0,45083.0,4.508302e+10,4.508302e+11,16448.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,10.0,12,4.503660e+11
67093,67093,3,823,49,84256.0,45.0,45083.0,4.508302e+10,4.508302e+11,26074.0,...,0.0,0.0,0.0,0.0,1.0,1.0,2.0,10.0,9,4.503540e+11
70547,70547,4,383,52,230407.0,45.0,45083.0,4.508302e+10,4.508302e+11,27518.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1.0,10.0,12,4.503480e+11
71901,71901,3,38,54,84256.0,45.0,45083.0,4.508302e+10,4.508302e+11,28085.0,...,0.0,0.0,0.0,0.0,1.0,1.0,2.0,10.0,9,4.503510e+11
246537,246537,3,116,163,84256.0,45.0,45083.0,4.508302e+10,4.508302e+11,96793.0,...,0.0,0.0,0.0,0.0,1.0,1.0,2.0,10.0,9,4.503660e+11
259863,259863,4,161,169,230407.0,45.0,45083.0,4.508302e+10,4.508302e+11,102071.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1.0,10.0,12,4.503480e+11


## Why does Gephi show between school connections?

The school network is made with stochastic block modeling (SMB). The SBM does not draw edges between schools. In networks.py it runs separately on each school’s student+worker list (sp_from_groups around lines 223–225). Mean degree 12 and assortativity 0.9 by grade.

A person who is a student at school A and a worker at school B is in both SBM groups, so they get connections in A and connections in B. Gephi then draws that one node as a bridge between two school clusters. Those are still within-school edges, not A–B student–student edges.

Use net_s plus existing membership tables to check:

In [20]:
net_s = pd.read_csv('pops/sc_45083/pop_export/starsim/net_s.csv')

# Schools each uid belongs to (student and/or worker)
stu = sch_students.dropna(subset=['uid'])[['uid', 'sch_code']].drop_duplicates()
wrk = sch_workers.dropna(subset=['uid'])[['uid', 'sch_code']].drop_duplicates()
membership = pd.concat([stu, wrk]).drop_duplicates()
uid_schools = membership.groupby('uid')['sch_code'].agg(set)

dual_uids = set(diff_school['uid'])

net = net_s.copy()
net['schools_p1'] = net['p1'].map(uid_schools)
net['schools_p2'] = net['p2'].map(uid_schools)

def share_school(a, b):
    if not isinstance(a, set) or not isinstance(b, set):
        return False
    return len(a & b) > 0

net['share_school'] = [
    share_school(a, b) for a, b in zip(net['schools_p1'], net['schools_p2'])
]
net['touches_dual'] = net['p1'].isin(dual_uids) | net['p2'].isin(dual_uids)

print('edges with no shared school (unexpected):', (~net['share_school']).sum())
print('edges incident to a dual-role uid:', net['touches_dual'].sum())

edges with no shared school (unexpected): 0
edges incident to a dual-role uid: 228


If share_school is true for almost every edge, the SBM is within-school as designed. The Gephi between-cluster links should then be the touches_dual edges: one endpoint is the student–worker, the other is someone at their other school.

To mimic Gephi (one cluster per node), assign each uid a single school, then count “cross-cluster” edges that hit a dual-role person:

In [21]:
# Prefer student school; if none, worker school
layout_school = (
    stu.drop_duplicates('uid')
    .set_index('uid')['sch_code']
    .combine_first(wrk.drop_duplicates('uid').set_index('uid')['sch_code'])
)

net['cluster_p1'] = net['p1'].map(layout_school)
net['cluster_p2'] = net['p2'].map(layout_school)
cross = net[net['cluster_p1'] != net['cluster_p2']].copy()

print('cross-cluster edges:', len(cross))
print('of those, incident to dual-role uid:', cross['touches_dual'].sum())
print('share unexplained by dual-role:', (~cross['touches_dual']).sum())
cross.head()

cross-cluster edges: 106
of those, incident to dual-role uid: 106
share unexplained by dual-role: 0


,p1,p2,edge_weight,schools_p1,schools_p2,share_school,touches_dual,cluster_p1,cluster_p2
562,5602,239,1,{450366001046},"{450366001052, 450366001046}",True,True,450366001046,450366001052
563,5772,239,1,{450366001046},"{450366001052, 450366001046}",True,True,450366001046,450366001052
564,239,5971,1,"{450366001052, 450366001046}",{450366001046},True,True,450366001052,450366001046
567,74833,239,1,{450366001046},"{450366001052, 450366001046}",True,True,450366001046,450366001052
569,163344,239,1,{450366001046},"{450366001052, 450366001046}",True,True,450366001046,450366001052


If nearly all cross edges have touches_dual == True, the between-school links in Gephi are the student-in-one-school / worker-in-another agents.

In [22]:
# 1) Real between-school test: do p1 and p2 share ANY school?
print('total edges', len(net))
print('share_school', net['share_school'].sum())
print('no shared school', (~net['share_school']).sum())
print('unmapped p1', net['schools_p1'].isna().sum())
print('unmapped p2', net['schools_p2'].isna().sum())

# 2) Why layout_school disagreed
print('cluster_p1 dtype', net['cluster_p1'].dtype, 'cluster_p2 dtype', net['cluster_p2'].dtype)
print('cluster_p1 missing', net['cluster_p1'].isna().sum())
print('cluster_p2 missing', net['cluster_p2'].isna().sum())
print(net[['p1','p2','cluster_p1','cluster_p2']].head(10))

# 3) Same school, different types?
net['same_school_as_str'] = (
    net['cluster_p1'].astype(str) == net['cluster_p2'].astype(str)
)
print('cross if compared as string', (~net['same_school_as_str']).sum())

total edges 351590
share_school 351590
no shared school 0
unmapped p1 0
unmapped p2 0
cluster_p1 dtype int64 cluster_p2 dtype int64
cluster_p1 missing 0
cluster_p2 missing 0
       p1      p2    cluster_p1    cluster_p2
0       1   20877  450363001032  450363001032
1   86107       1  450363001032  450363001032
2       1   86637  450363001032  450363001032
3       1   91188  450363001032  450363001032
4   92038       1  450363001032  450363001032
5       1  159939  450363001032  450363001032
6  160502       1  450363001032  450363001032
7  220473       1  450363001032  450363001032
8  220957       1  450363001032  450363001032
9       1  238305  450363001032  450363001032
cross if compared as string 106


*Note: the values below (and in the code comments above) are from one run of the population generation for this county. They will differ if the population is regenerated.*

Every one of the 351,590 edges has both endpoints in the same school (share_school = 351,590, none unmapped). 

The 106 string-mismatch edges are the leftover dual-role cases: a student at school A who also works at school B gets one layout school (usually the student school), so their worker-side edges look cross-cluster. That is about 0.03% of edges, and it lines up with the 9 people who were students in one school and workers in another.

Gephi may still show a few thin bridges between school blobs; those should be those 9 people, not a second between-school process. Confirm with cross['touches_dual'].mean() — it is 1.0, so all 106 cross-cluster edges touch a dual-role agent.

In [15]:
cross['touches_dual'].mean() 

np.float64(1.0)

## Make sure student-workers are represented in at least one network (s, w, or g)

In [23]:
# uids in school network
network_uids_s = set(net_s.p1).union(set(net_s.p2))
network_uids_s

# k-12 workers
k12_workers = student_workers[~student_workers['sch_grade'].isin(['g', 'c'])]

# all k-12 workers in school network
print('number of k-12 workers: ', k12_workers.shape[0])
print('number of k-12 workers in school network: ', k12_workers['uid'].isin(network_uids_s).sum())


number of k-12 workers:  2759
number of k-12 workers in school network:  2759


In [24]:
cg_workers = student_workers[student_workers['sch_grade'].isin(['g', 'c'])]
print('number of workers in college or graduate school: ', cg_workers.shape[0])
print('number of college or graduate school workers in school network: ', cg_workers['uid'].isin(network_uids_s).sum())
# These will be workers in the school network not students because c and g are not grades in the school network

number of workers in college or graduate school:  9949
number of college or graduate school workers in school network:  168


In [25]:
# uids in workplace network
net_w = pd.read_csv('pops/sc_45083/pop_export/starsim/net_w.csv')
network_uids_w = set(net_w.p1).union(set(net_w.p2))
# print k12 workers in workplace network
print('number of k12 workers in workplace network: ', k12_workers['uid'].isin(network_uids_w).sum())
# print cg workers in workplace network
print('number of cg workers in workplace network: ', cg_workers['uid'].isin(network_uids_w).sum())

# uids in gq network
net_g = pd.read_csv('pops/sc_45083/pop_export/starsim/net_g.csv')
network_uids_g = set(net_g.p1).union(set(net_g.p2))
print('number of k12 workers in gq network: ', k12_workers['uid'].isin(network_uids_g).sum())
print('number of cg workers in gq network: ', cg_workers['uid'].isin(network_uids_g).sum())


number of k12 workers in workplace network:  1513
number of cg workers in workplace network:  5137
number of k12 workers in gq network:  1
number of cg workers in gq network:  22
